In [10]:
import pandas as pd
import os
import urllib.request
from datetime import datetime

# Словник для переведення англійського коду NOAA в українську назву
regions_dict = {
    1: 'Черкаська', 2: 'Чернігівська', 3: 'Чернівецька', 4: 'Крим', 
    5: 'Дніпропетровська', 6: 'Донецька', 7: 'Івано-Франківська', 8: 'Харківська', 
    9: 'Херсонська', 10: 'Хмельницька', 11: 'Київська', 12: 'Кіровоградська', 
    13: 'Луганська', 14: 'Львівська', 15: 'Миколаївська', 16: 'Одеська', 
    17: 'Полтавська', 18: 'Рівненська', 19: 'Севастополь', 20: 'Сумська', 
    21: 'Тернопільська', 22: 'Вінницька', 23: 'Волинська', 24: 'Закарпатська', 
    25: 'Запорізька', 26: 'Житомирська', 27: 'Київ'
}

# Новий реєстр: області за українським алфавітом (Вінницька = 1)
ua_regions = {
    1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 
    5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 
    9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська', 
    13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська', 
    17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська', 
    21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 
    25: 'Крим', 26: 'Київ', 27: 'Севастополь'
}

def prepare_vhi_data():
    if not os.path.exists('vhi_data'):
        os.makedirs('vhi_data')
    
    # 1. ЗАВАНТАЖЕННЯ (urllib)
    for i in range(1, 28):
        existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{i}_")]
        
        if not existing_files:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_path = f"vhi_data/vhi_id_{i}_{timestamp}.csv"
            url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
            print(f"Завантаження реальних даних для Province {i} з NOAA...")
            try:
                urllib.request.urlretrieve(url, file_path)
            except Exception as e:
                print(f"Помилка завантаження ID {i}: {e}")
        
    # 2. ОЧИЩЕННЯ ВІД HTML-ТЕГІВ ТА ПЕРЕІНДЕКСАЦІЯ
    all_frames = []
    ua_name_to_new_id = {v: k for k, v in ua_regions.items()}
    
    for i in range(1, 28):
        files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{i}_")]
        if files:
            f_p = os.path.join('vhi_data', files[0])
            
            valid_rows = []
            with open(f_p, 'r') as file:
                for line in file:
                    # Повністю ігноруємо html сміття
                    if '<' in line or '>' in line or 'html' in line:
                        continue
                    parts = line.strip().split(',')
                    # Перевіряємо чи рядок містить потрібну нам кількість колонок і чи перший елемент — це рік
                    if len(parts) >= 7 and parts[0].strip().isdigit():
                        valid_rows.append([float(x.strip()) for x in parts[:7]])
            
            if not valid_rows:
                continue
                
            df_tmp = pd.DataFrame(valid_rows, columns=['Year','Week','SMN','SMT','VCI','TCI','VHI'])
            df_tmp['Year'] = df_tmp['Year'].astype(int)
            df_tmp['Week'] = df_tmp['Week'].astype(int)
            
            # Заміна індексів відповідно до українського алфавіту
            old_name = regions_dict.get(i)
            new_id = ua_name_to_new_id.get(old_name)
            
            df_tmp['Area_ID'] = new_id
            df_tmp['Area'] = old_name
            
            print(f" > Успішно зчитано рядків для {old_name}: {len(df_tmp)}")
            all_frames.append(df_tmp)
    
    if not all_frames:
        print("[!] Критична помилка: не вдалося знайти числові дані у файлах.")
        return pd.DataFrame()
        
    master_df = pd.concat(all_frames, ignore_index=True)
    return master_df

def get_vhi_extremes(df, area, year):
    filt = df[(df['Area'] == area) & (df['Year'] == year)]
    if filt.empty: 
        return (None, None, None, None)
    return (filt['VHI'].max(), filt['VHI'].min(), filt['VHI'].mean(), filt['VHI'].median())

def get_droughts(df, area, mode='extreme'):
    if mode == 'extreme':
        filt = df[(df['Area'] == area) & (df['VHI'] < 15)]
    else:
        filt = df[(df['Area'] == area) & (df['VHI'] >= 15) & (df['VHI'] <= 35)]
    return filt['Year'].unique()

# Вивід результатів
print("\n" + "="*60)
print("ЛАБОРАТОРНА РОБОТА №2 — ЧАСТИНА 1 (ОНОВЛЕНА)")
print("="*60)

df_vhi = prepare_vhi_data()

if not df_vhi.empty:
    print("\n--- Загальний розмір об'єднаного датасету: ---", df_vhi.shape)
    print("\n--- Перші 5 рядків об'єднаної таблиці: ---")
    print(df_vhi[['Year', 'Week', 'VHI', 'Area_ID', 'Area']].head())

    area_name = 'Київська'
    target_year = 2020
    v_max, v_min, v_mean, v_med = get_vhi_extremes(df_vhi, area_name, target_year)
    
    print(f"\n--- Результати для області: {area_name} ({target_year} рік) ---")
    if v_max is not None:
        print(f" > Максимальний VHI: {v_max:.2f} | Мінімальний VHI: {v_min:.2f}")
        print(f" > Середнє: {v_mean:.2f} | Медіана: {v_med:.2f}")
    else:
        print(" Дані за вказаний рік або область відсутні в базі.")

    ex_years = get_droughts(df_vhi, area_name, 'extreme')
    print(f"\n--- Роки з екстремальними посухами (VHI < 15): ---")
    print(ex_years if len(ex_years) > 0 else "Посух не знайдено")
else:
    print("[!] Фінальний датафрейм порожній. Будь ласка, видаліть папку vhi_data та запустіть код знову.")


ЛАБОРАТОРНА РОБОТА №2 — ЧАСТИНА 1 (ОНОВЛЕНА)
Завантаження реальних даних для Province 1 з NOAA...
Завантаження реальних даних для Province 2 з NOAA...
Завантаження реальних даних для Province 3 з NOAA...
Завантаження реальних даних для Province 4 з NOAA...
Завантаження реальних даних для Province 5 з NOAA...
Завантаження реальних даних для Province 6 з NOAA...
Завантаження реальних даних для Province 7 з NOAA...
Завантаження реальних даних для Province 8 з NOAA...
Завантаження реальних даних для Province 9 з NOAA...
Завантаження реальних даних для Province 10 з NOAA...
Завантаження реальних даних для Province 11 з NOAA...
Завантаження реальних даних для Province 12 з NOAA...
Завантаження реальних даних для Province 13 з NOAA...
Завантаження реальних даних для Province 14 з NOAA...
Завантаження реальних даних для Province 15 з NOAA...
Завантаження реальних даних для Province 16 з NOAA...
Завантаження реальних даних для Province 17 з NOAA...
Завантаження реальних даних для Province 18 з